# A tiny Dreamer-V2-style RSSM for CartPole

This notebook builds the smallest useful recurrent state-space model (RSSM) for a Dreamer-V2-style world model.

The full CartPole state is `[cart_position, cart_velocity, pole_angle, pole_angular_velocity]`. We remove the two velocity entries and give the model only `[cart_position, pole_angle]`. The recurrent state has to remember recent observations and actions if it wants to infer the missing velocity information.

There is no image encoder and no decoder here. The posterior receives the raw 2D observation directly, and the prediction head predicts the next raw 2D observation directly.

The Dreamer-V2-style change is the latent state: instead of a Gaussian vector, the stochastic state is a few independent categorical variables. Each categorical variable is represented as a one-hot vector, and the model uses a straight-through sample so the forward pass is discrete while gradients can still flow through the probabilities.


## Imports

The code follows the same compact PyTorch style as `pytorch_intro.ipynb`: import the core pieces, pick a device, define an `nn.Module`, then write explicit `train` and `test` functions.


In [1]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.distributions import Normal
from torch.utils.data import DataLoader, Dataset

try:
    import gymnasium as gym
except ImportError:
    import gym


In [2]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if hasattr(torch, "accelerator") and torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator().type
elif torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device")


Using mps device


## CartPole with hidden velocities

CartPole normally returns four numbers. We keep only indices `0` and `2`, so the model sees cart position and pole angle but not either velocity.

Actions are represented as one-hot vectors: `[1, 0]` for pushing left and `[0, 1]` for pushing right.


In [3]:
GAME_NAME = "CartPole-v1"
VISIBLE_STATE_INDICES = np.array([0, 2])
VISIBLE_STATE_NAMES = ["cart position", "pole angle"]


def make_env(seed):
    env = gym.make(GAME_NAME)
    env.action_space.seed(seed)
    return env


def reset_env(env, seed=None):
    result = env.reset(seed=seed) if seed is not None else env.reset()
    observation = result[0] if isinstance(result, tuple) else result
    return np.asarray(observation, dtype=np.float32)


def step_env(env, action):
    result = env.step(action)
    if len(result) == 5:
        observation, reward, terminated, truncated, _ = result
        done = terminated or truncated
    else:
        observation, reward, done, _ = result
    return np.asarray(observation, dtype=np.float32), float(reward), bool(done)


def visible_state(full_state):
    return full_state[VISIBLE_STATE_INDICES].astype(np.float32)


def one_hot(action, action_size=2):
    action_vector = np.zeros(action_size, dtype=np.float32)
    action_vector[action] = 1.0
    return action_vector


env = make_env(seed)
full_state = reset_env(env, seed=seed)
print(f"Full CartPole state: {full_state}")
print(f"Visible model input: {visible_state(full_state)}")
env.close()


Full CartPole state: [ 0.01369617 -0.02302133 -0.04590265 -0.04834723]
Visible model input: [ 0.01369617 -0.04590265]


## Random episodes become supervised sequences

The RSSM is only a world model. It does not choose good actions. We collect random CartPole episodes and train the model to predict what happened next.

Each training sample is a short chunk:

- observations: shape `[sequence + 1, 2]`
- actions: shape `[sequence, 2]`
- rewards: shape `[sequence, 1]`
- continues: shape `[sequence, 1]`, where `0` means the episode ended after that step


In [4]:
def collect_episodes(env, episodes, seed):
    collected = []

    for episode in range(episodes):
        full_state = reset_env(env, seed=seed + episode)
        observations = [visible_state(full_state)]
        full_states = [full_state]
        actions, rewards, continues = [], [], []
        done = False

        while not done:
            action = env.action_space.sample()
            next_full_state, reward, done = step_env(env, action)

            actions.append(one_hot(action))
            rewards.append([reward])
            continues.append([0.0 if done else 1.0])
            observations.append(visible_state(next_full_state))
            full_states.append(next_full_state)

        collected.append(
            {
                "observations": np.asarray(observations, dtype=np.float32),
                "full_states": np.asarray(full_states, dtype=np.float32),
                "actions": np.asarray(actions, dtype=np.float32),
                "rewards": np.asarray(rewards, dtype=np.float32),
                "continues": np.asarray(continues, dtype=np.float32),
            }
        )

    return collected


class SequenceDataset(Dataset):
    def __init__(self, episodes, sequence_length):
        self.episodes = episodes
        self.sequence_length = sequence_length
        self.index = []

        for episode_index, episode in enumerate(episodes):
            max_start = len(episode["actions"]) - sequence_length
            for start in range(max_start + 1):
                self.index.append((episode_index, start))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, index):
        episode_index, start = self.index[index]
        episode = self.episodes[episode_index]
        end = start + self.sequence_length

        return {
            "observations": torch.tensor(episode["observations"][start : end + 1]),
            "actions": torch.tensor(episode["actions"][start:end]),
            "rewards": torch.tensor(episode["rewards"][start:end]),
            "continues": torch.tensor(episode["continues"][start:end]),
        }


In [5]:
sequence_length = 16
batch_size = 64

# Collect data once. The learning problem below is supervised sequence prediction.
env = make_env(seed)
episodes = collect_episodes(env, episodes=240, seed=seed)
env.close()

split = int(0.8 * len(episodes))
train_episodes = episodes[:split]
test_episodes = episodes[split:]

train_data = SequenceDataset(train_episodes, sequence_length)
test_data = SequenceDataset(test_episodes, sequence_length)
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

transition_count = sum(len(episode["actions"]) for episode in episodes)
print(f"Collected {len(episodes)} episodes and {transition_count} transitions.")
print(f"Training chunks: {len(train_data)}, test chunks: {len(test_data)}")

for batch in train_dataloader:
    print(f"observations: {batch['observations'].shape}")
    print(f"actions:      {batch['actions'].shape}")
    print(f"rewards:      {batch['rewards'].shape}")
    print(f"continues:    {batch['continues'].shape}")
    break


Collected 240 episodes and 5464 transitions.
Training chunks: 1582, test chunks: 465
observations: torch.Size([64, 17, 2])
actions:      torch.Size([64, 16, 2])
rewards:      torch.Size([64, 16, 1])
continues:    torch.Size([64, 16, 1])


## RSSM parts

The RSSM has four moving pieces.

`deterministic_state` is the GRU memory. It stores information that can be inferred from the history, such as velocity hints from changes in position and angle.

`prior_model` predicts categorical logits from the GRU memory only. This is the model used when imagining forward without seeing the next observation.

`posterior_model` predicts categorical logits from the GRU memory and the next real observation. This is used during training as a teacher-forced latent state.

`stochastic_state` is a flattened stack of one-hot categorical samples. In this notebook it uses `4` categorical groups with `4` classes each, so the flattened latent has `16` entries.

`prediction_model` predicts the next visible observation, reward, and continue flag from the deterministic and stochastic states.


In [ ]:
class TinyRSSM(nn.Module):
    def __init__(
        self,
        observation_size=2,
        action_size=2,
        stochastic_groups=4,
        classes_per_group=4,
        deterministic_size=8,
        hidden_size=16,
        min_std=0.05,
    ):
        super().__init__()
        self.stochastic_groups = stochastic_groups
        self.classes_per_group = classes_per_group
        self.stochastic_size = stochastic_groups * classes_per_group
        self.deterministic_size = deterministic_size
        self.min_std = min_std

        self.action_stack = nn.Sequential(
            nn.Linear(self.stochastic_size + action_size, hidden_size),
            nn.ReLU(),
        )
        self.rnn = nn.GRUCell(hidden_size, deterministic_size)

        self.prior_model = nn.Sequential(
            nn.Linear(deterministic_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, self.stochastic_size),
        )
        self.posterior_model = nn.Sequential(
            nn.Linear(deterministic_size + observation_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, self.stochastic_size),
        )

        self.prediction_model = nn.Sequential(
            nn.Linear(deterministic_size + self.stochastic_size, hidden_size),
            nn.ReLU(),
        )
        self.observation_head = nn.Linear(hidden_size, 2 * observation_size)
        self.reward_head = nn.Linear(hidden_size, 2)
        self.continue_head = nn.Linear(hidden_size, 1)

    def categorical_logits(self, parameters):
        return parameters.view(
            parameters.shape[0], self.stochastic_groups, self.classes_per_group
        )

    def flatten_stochastic(self, stochastic_state):
        return stochastic_state.reshape(stochastic_state.shape[0], self.stochastic_size)

    def straight_through_sample(self, logits):
        probabilities = F.softmax(logits, dim=-1)
        index = torch.distributions.Categorical(logits=logits).sample()
        sample = F.one_hot(index, self.classes_per_group).float()
        straight_through = probabilities + (sample - probabilities).detach()
        return self.flatten_stochastic(straight_through)

    def mode(self, logits):
        index = logits.argmax(dim=-1)
        sample = F.one_hot(index, self.classes_per_group).float()
        return self.flatten_stochastic(sample)

    def initial(self, batch_size, device):
        stochastic_state = torch.zeros(batch_size, self.stochastic_size, device=device)
        deterministic_state = torch.zeros(batch_size, self.deterministic_size, device=device)
        return stochastic_state, deterministic_state

    def prediction_heads(self, stochastic_state, deterministic_state):
        features = self.prediction_model(
            torch.cat([stochastic_state, deterministic_state], dim=-1)
        )
        observation_mean, observation_raw_std = self.observation_head(features).chunk(2, dim=-1)
        reward_mean, reward_raw_std = self.reward_head(features).chunk(2, dim=-1)

        return {
            "observation_mean": observation_mean,
            "observation_std": F.softplus(observation_raw_std) + self.min_std,
            "reward_mean": reward_mean,
            "reward_std": F.softplus(reward_raw_std) + self.min_std,
            "continue_logit": self.continue_head(features),
        }

    def observe(self, observations, actions):
        batch_size, sequence_plus_one, _ = observations.shape
        sequence_length = sequence_plus_one - 1
        stochastic_state, deterministic_state = self.initial(batch_size, observations.device)

        prior_logits = []
        posterior_logits = []
        prior_predictions = []
        posterior_predictions = []

        for t in range(sequence_length):
            action_features = self.action_stack(
                torch.cat([stochastic_state, actions[:, t]], dim=-1)
            )
            deterministic_state = self.rnn(action_features, deterministic_state)

            prior = self.categorical_logits(self.prior_model(deterministic_state))
            posterior = self.categorical_logits(
                self.posterior_model(
                    torch.cat([deterministic_state, observations[:, t + 1]], dim=-1)
                )
            )

            prior_stochastic_state = self.straight_through_sample(prior)
            stochastic_state = self.straight_through_sample(posterior)

            prior_logits.append(prior)
            posterior_logits.append(posterior)
            prior_predictions.append(
                self.prediction_heads(prior_stochastic_state, deterministic_state)
            )
            posterior_predictions.append(
                self.prediction_heads(stochastic_state, deterministic_state)
            )

        return {
            "prior_logits": prior_logits,
            "posterior_logits": posterior_logits,
            "prior_predictions": stack_predictions(prior_predictions),
            "posterior_predictions": stack_predictions(posterior_predictions),
        }

    def forward(self, observations, actions):
        return self.observe(observations, actions)


## Loss terms

The posterior predictions are easy mode: the model has just seen the real next observation through the posterior.

The prior predictions are the useful mode: the model predicts from history and action only.

The KL term keeps the posterior close to the prior, so the prior can be used for imagination after training. With categorical latents, the KL is just the KL between the posterior and prior class probabilities in each group.

The printed loss can become negative. That is fine here because the observation and reward losses are negative log probabilities under continuous Gaussian densities, and a density value can be larger than one.


In [ ]:
def stack_predictions(predictions):
    return {
        key: torch.stack([prediction[key] for prediction in predictions], dim=1)
        for key in predictions[0]
    }


def prediction_loss(prediction, next_observations, rewards, continues):
    observation_dist = Normal(prediction["observation_mean"], prediction["observation_std"])
    reward_dist = Normal(prediction["reward_mean"], prediction["reward_std"])

    observation_loss = -observation_dist.log_prob(next_observations).sum(dim=-1).mean()
    reward_loss = -reward_dist.log_prob(rewards).sum(dim=-1).mean()
    continue_loss = F.binary_cross_entropy_with_logits(
        prediction["continue_logit"],
        continues,
    )
    loss = observation_loss + reward_loss + continue_loss

    with torch.no_grad():
        continue_probability = torch.sigmoid(prediction["continue_logit"])
        metrics = {
            "observation_mse": F.mse_loss(
                prediction["observation_mean"], next_observations
            ).item(),
            "reward_mse": F.mse_loss(prediction["reward_mean"], rewards).item(),
            "continue_accuracy": (
                ((continue_probability >= 0.5).float() == continues)
                .float()
                .mean()
                .item()
            ),
        }

    return loss, metrics


def categorical_kl(posterior_logits, prior_logits):
    posterior_log_probs = F.log_softmax(posterior_logits, dim=-1)
    prior_log_probs = F.log_softmax(prior_logits, dim=-1)
    posterior_probs = posterior_log_probs.exp()
    kl = posterior_probs * (posterior_log_probs - prior_log_probs)
    return kl.sum(dim=(-1, -2)).mean()


def rssm_loss(model, observations, actions, rewards, continues, kl_scale=0.1, prior_scale=0.5):
    output = model(observations, actions)
    next_observations = observations[:, 1:]

    posterior_loss, posterior_metrics = prediction_loss(
        output["posterior_predictions"], next_observations, rewards, continues
    )
    prior_loss, prior_metrics = prediction_loss(
        output["prior_predictions"], next_observations, rewards, continues
    )

    kl_values = [
        categorical_kl(posterior, prior)
        for posterior, prior in zip(output["posterior_logits"], output["prior_logits"])
    ]
    kl_loss = torch.stack(kl_values).mean()
    loss = posterior_loss + prior_scale * prior_loss + kl_scale * kl_loss

    metrics = {
        "loss": loss.item(),
        "kl": kl_loss.item(),
        "posterior_observation_mse": posterior_metrics["observation_mse"],
        "observation_mse": prior_metrics["observation_mse"],
        "reward_mse": prior_metrics["reward_mse"],
        "continue_accuracy": prior_metrics["continue_accuracy"],
    }
    return loss, metrics


In [ ]:
model = TinyRSSM().to(device)
print(model)
print(
    f"Categorical latent: {model.stochastic_groups} groups x "
    f"{model.classes_per_group} classes = {model.stochastic_size} one-hot entries"
)
print(f"Parameters: {sum(parameter.numel() for parameter in model.parameters())}")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


## Training and testing

The loop mirrors the PyTorch intro notebook: switch to train mode, compute the prediction error, backpropagate, update the optimizer, then test in `torch.no_grad()` mode.


In [ ]:
def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def train(dataloader, model, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch_index, batch in enumerate(dataloader):
        batch = move_batch(batch, device)

        # Compute prediction error
        loss, metrics = rssm_loss(
            model,
            batch["observations"],
            batch["actions"],
            batch["rewards"],
            batch["continues"],
        )

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch_index % 50 == 0:
            current = min((batch_index + 1) * batch_size, size)
            print(
                f"loss: {metrics['loss']:>7f}  "
                f"obs_mse: {metrics['observation_mse']:.5f}  "
                f"kl: {metrics['kl']:.4f}  "
                f"[{current:>5d}/{size:>5d}]"
            )


def test(dataloader, model):
    num_batches = len(dataloader)
    model.eval()
    totals = {
        "loss": 0.0,
        "kl": 0.0,
        "posterior_observation_mse": 0.0,
        "observation_mse": 0.0,
        "reward_mse": 0.0,
        "continue_accuracy": 0.0,
    }

    with torch.no_grad():
        for batch in dataloader:
            batch = move_batch(batch, device)
            _, metrics = rssm_loss(
                model,
                batch["observations"],
                batch["actions"],
                batch["rewards"],
                batch["continues"],
            )
            for key in totals:
                totals[key] += metrics[key]

    for key in totals:
        totals[key] /= num_batches

    print(
        "Test Error: \n"
        f" Prior obs MSE: {totals['observation_mse']:>8f}, "
        f"Posterior obs MSE: {totals['posterior_observation_mse']:>8f}, "
        f"Reward MSE: {totals['reward_mse']:>8f}, "
        f"Continue acc: {(100 * totals['continue_accuracy']):>0.1f}%, "
        f"KL: {totals['kl']:>8f} \n"
    )
    return totals


In [ ]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train(train_dataloader, model, optimizer)
    test(test_dataloader, model)
print("Done!")


## Prior rollout

The posterior is useful for training, but a world model must also run without seeing future observations.

The next cell warms up the latent state with a few real observations, then switches to the prior. During the prior rollout, the model receives only the recorded actions and its own latent state. The target curves are the real hidden-velocity CartPole trajectory with only the visible coordinates plotted.


In [ ]:
def prior_rollout(model, episode, warmup_steps=5, rollout_steps=40):
    model.eval()
    observations = torch.tensor(episode["observations"], dtype=torch.float32, device=device).unsqueeze(0)
    actions = torch.tensor(episode["actions"], dtype=torch.float32, device=device).unsqueeze(0)

    max_rollout = min(rollout_steps, actions.shape[1] - warmup_steps)
    stochastic_state, deterministic_state = model.initial(1, observations.device)

    with torch.no_grad():
        for t in range(warmup_steps):
            action_features = model.action_stack(
                torch.cat([stochastic_state, actions[:, t]], dim=-1)
            )
            deterministic_state = model.rnn(action_features, deterministic_state)
            posterior = model.categorical_logits(
                model.posterior_model(
                    torch.cat([deterministic_state, observations[:, t + 1]], dim=-1)
                )
            )
            stochastic_state = model.mode(posterior)

        predicted_observations = []
        for t in range(warmup_steps, warmup_steps + max_rollout):
            action_features = model.action_stack(
                torch.cat([stochastic_state, actions[:, t]], dim=-1)
            )
            deterministic_state = model.rnn(action_features, deterministic_state)
            prior = model.categorical_logits(model.prior_model(deterministic_state))
            stochastic_state = model.mode(prior)
            prediction = model.prediction_heads(stochastic_state, deterministic_state)
            predicted_observations.append(prediction["observation_mean"].squeeze(0).cpu())

    predicted_observations = torch.stack(predicted_observations).numpy()
    target_observations = episode["observations"][warmup_steps + 1 : warmup_steps + 1 + max_rollout]
    return predicted_observations, target_observations


long_episode = max(test_episodes, key=lambda episode: len(episode["actions"]))
predicted, target = prior_rollout(model, long_episode)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for index, ax in enumerate(axes):
    ax.plot(target[:, index], label="real")
    ax.plot(predicted[:, index], label="prior prediction")
    ax.set_title(VISIBLE_STATE_NAMES[index])
    ax.set_xlabel("step after warmup")
    ax.grid(True, alpha=0.3)
axes[0].legend()
plt.tight_layout()
plt.show()


## Hidden variables check

These two curves are the velocity variables that were never given to the model. The RSSM can only infer their effect indirectly from observation history and actions.


In [ ]:
full_states = long_episode["full_states"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(full_states[:, 1])
axes[0].set_title("cart velocity, hidden")
axes[1].plot(full_states[:, 3])
axes[1].set_title("pole angular velocity, hidden")

for ax in axes:
    ax.set_xlabel("environment step")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## What this leaves out

This is Dreamer-V2-like because it learns a latent recurrent world model with prior and posterior categorical latent states.

It is not a full Dreamer-V2 agent: there is no actor, critic, imagination objective, image encoder, image decoder, KL balancing, or replay-based online data collection. The point of the notebook is to isolate the categorical RSSM mechanics before adding policy learning.
